# KMX MTN-Specific RAGU Score Notebook
This notebook computes RAGU scores for KMX loans, broken down by MTN model (3.0, 3.1, 3.2, 4.1).
Aligned with `bareboned_ragu_new.ipynb` for diagnostic purposes.
- **Granularity:** Configurable (weekly/monthly/quarterly) via `granularity` in cell 1
- **Date handling:** Weekly uses `app_date`, monthly/quarterly use `book_date`
- **Output:** Per-model RAGU decomposition + diagnostics, exported to `new_kmx_models.xlsx`

## How to Run
1. Set `granularity`, `START_DATE`, `END_DATE` in cell 1
2. Set `run_from_pickle = True` to load pre-computed data (faster), or `False` to re-run SQL queries
3. Run All Cells
4. Results are exported to `new_kmx_models.xlsx`

In [1]:
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)
import openpyxl
import datetime as dt
import re
import os

# ── Configuration ──
granularity = 'q'                # 'q' = quarterly, 'm' = monthly, 'w' = weekly

START_DATE = '2023-01-01'
END_DATE = None                  # None = auto-detect from today's date

run_every_query = True           # True = re-run SQL, False = load pickle
run_from_pickle = not run_every_query

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}
PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}

BASELINES = {
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'kmx_loss_scale': 0.067,
}

MTN_MODELS = [3.0, 3.1, 3.2, 4.1]
EXCEL_OUTPUT = '../output/new_kmx_models.xlsx'

# ── Derived values ──
date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()
period_freq = PERIOD_FREQ_MAP[granularity]
start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)
min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: q
Date column: book_date
Period range: 2023Q1 to 2026Q3
SQL min_date: '2023-01-01'


In [2]:
def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    """Fetch from SQL and cache to pickle. Reuse cache unless force_refresh=True
    or the pickle file is missing."""
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings."""
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)


def rebuild_ms_df(ula_subset):
    """Recompute ms_df from a filtered ula_df subset (for per-model scoring).
    MTN 4.1 transform is already applied in-place on ula_df_total upstream."""
    ms_source = ula_subset[['lob', 'cd_model_score', 'amt_financed', 'period']].copy()
    ms_source = ms_source.rename(columns={'cd_model_score': 'model_score'})
    ms = ms_source.groupby(['period', 'lob']).apply(
        weighted_average_and_sum, 'model_score', include_groups=False
    ).reset_index()
    ms['period'] = format_vintage(ms['period'])
    return ms

In [3]:
# Per-table caches under cache/ with _v1 schema tags. When run_from_pickle=True
# (i.e. run_every_query=False) each cached_sql call reuses its pickle if present
# and falls through to SQL if missing. Delete an individual pickle to force a
# selective refresh.
#
# Note: ULA is cached under cache/ula_kmx_v1.pkl because this notebook
# applies a KMX-only filter via sub_list; that is a different schema (KMX
# subset of rows) than the full-LOB cache/ula_v1.pkl written by
# bareboned_ragu_new.ipynb. DLA and new_recovery use identical queries in both
# notebooks, so they share the same cache files.

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/ula_kmx_v1.pkl', '../../cache/dla_v1.pkl', '../../cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            '../queries/vintage_level_ula_query.txt', '../../cache/ula_kmx_v1.pkl',
            sub_list=[('{min_book_date}', f"{min_date_sql}\n  AND dru.riskdealergroup = 'KMX'")],
            connection=conn, force_refresh=force,
        )
        print('ULA ready')

        dla_df = cached_sql(
            '../queries/new_dll_query.txt', '../../cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')

        new_recovery = cached_sql(
            '../queries/new_recovery_queryt.txt', '../../cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('../../cache/ula_kmx_v1.pkl')
    dla_df = get_pickle('../../cache/dla_v1.pkl')
    new_recovery = get_pickle('../../cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")


ULA ready
DLA ready
New recovery ready
ULA records: 604,002
[PROGRESS] Data Fetch Complete


In [4]:
def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']
    ula_df['loss_multiplier'] = 1.0

    if leave_out!='Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)\
                                    + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag)

    if leave_out!='Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)\
                                    + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag)

    if leave_out!='High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0 * ula_df.normal_pti_flag \
                                    + 0.05 * ula_df.high_pti_tier_1_flag \
                                    + 0.1 * ula_df.high_pti_tier_2_flag \
                                    + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag

    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] /= (1 + loss_scale)

    if leave_out!='Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)\
                                        * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag)

    if leave_out!='Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out!='Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out!='Clip':
        clipped_multiplier = np.clip(ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
        ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'] = clipped_multiplier

    if leave_out!='Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier']  *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out !='npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.96 +  0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]),'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out!='Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier']  *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out!='georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier']  *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out!='txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140) ,'loss_multiplier']  *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) &  (ula_df.cd_model_score >= 135)   ,'loss_multiplier']  *= 1 - 0.05 * ula_df.txca_flag

    if leave_out!='state_counter_adj':
        ula_df.loc[ ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1])  & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag) ,'loss_multiplier'] *= 1.012

    if leave_out!='Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.96 + (0.01 * ula_df.soft_pull_flag  - 0.18*ula_df.chime_flag* ula_df.soft_pull_flag) + 0.46*ula_df.chime_flag

    if leave_out!='Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out!='Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out!='Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out!='Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out!='Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag,'loss_multiplier'] *= 1.1  * (0.99 + 0.11*ula_df.low_bureau_flag)  * (0.978 + 0.172*ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag,'loss_multiplier'] *= (1 * (0.98 + 0.22*ula_df.low_bureau_flag)  * (0.945 + 0.405*ula_df.open_tl_flag ) /np.maximum(ula_df.cd_perc_flag*ula_df.open_tl_flag*1.2,1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) &  ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97  + 0.15  * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1)         &  ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0   + 0.05  * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1)         & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0   + 0.10  * ula_df.cd_perc_flag


    if leave_out!='blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1


    if leave_out!='Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df


def get_ula_multiplier_kmx_diag(ula_df, loss_scale=None, leave_out='None', verbose=True):
    """Identical logic to get_ula_multiplier_kmx, but also returns step-by-step loss_multiplier means and flag means."""
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']
    steps = {}
    def _record(label, df):
        steps[label] = df.loss_multiplier.mean()

    ula_df['loss_multiplier'] = 1.0
    _record('00_initial', ula_df)
    if verbose:
        print(f"00_initial  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag) \
                                    + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag)
    _record('02_low_fico_3.0', ula_df)
    if verbose:
        print(f"02_low_fico | flag mean: {ula_df.low_fico_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag) \
                                    + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag)
    _record('03_low_vantage_3.0', ula_df)
    if verbose:
        print(f"03_low_vant | flag mean: {ula_df.low_vantage_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0 * ula_df.normal_pti_flag \
                                    + 0.05 * ula_df.high_pti_tier_1_flag \
                                    + 0.1 * ula_df.high_pti_tier_2_flag \
                                    + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag
    _record('04_high_pti_3.0', ula_df)
    if verbose:
        print(f"04_high_pti | tier1: {ula_df.high_pti_tier_1_flag.mean():.6f}  tier2: {ula_df.high_pti_tier_2_flag.mean():.6f}  tier3: {ula_df.high_pti_tier_3_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)
    _record('07_loss_scale_div_3.0', ula_df)
    if verbose:
        print(f"07_scale_dv | dividing by (1 + {loss_scale})  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + (-0.01 + 0.06 * ula_df.secured_credit_flag) \
                                        * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag)
    _record('08_secured_credit_3.0', ula_df)
    if verbose:
        print(f"08_sec_cred | flag mean: {ula_df.secured_credit_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag
    _record('09_auth_tradelines_3.0', ula_df)
    if verbose:
        print(f"09_auth_tl  | flag mean: {ula_df.kmx_auth_tradelines_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))
    _record('11_soft_pull_3.0', ula_df)
    if verbose:
        print(f"11_soft_pll | flag mean: {ula_df.soft_pull_flag.mean():.6f}  narrowed: {ula_df.narrowed_soft_pull_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)
    _record('12_fraud_all', ula_df)
    if verbose:
        print(f"12_fraud    | fraud_adj mean: {ula_df.fraud_adjustment.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Clip':
        clipped_multiplier = np.clip(ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
        ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'] = clipped_multiplier
    _record('13_clip_3.0', ula_df)
    if verbose:
        print(f"13_clip_3.0 | clip [0.8, {1.35 / (1 + loss_scale):.4f}]  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age
    _record('14_vehicle_age_3.0', ula_df)
    if verbose:
        print(f"14_veh_age  | continuous_age mean: {ula_df.continuous_vehicle_age.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc
    _record('15_npc_all', ula_df)
    if verbose:
        print(f"15_npc      | npc_flag mean: {ula_df.kmx_npc_flag.mean():.6f}  high_pti_npc mean: {ula_df.high_pti_npc.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag
    _record('16_student_loans_all', ula_df)
    if verbose:
        print(f"16_stu_loan | flag mean: {ula_df.student_loan_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag
    _record('17_high_sales_price_4.1', ula_df)
    if verbose:
        print(f"17_hi_price | flag mean: {ula_df.high_sales_price_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag
    _record('18_driver_flag_all', ula_df)
    if verbose:
        print(f"18_driver   | flag mean: {ula_df.driver_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag
    _record('19_louisiana_all', ula_df)
    if verbose:
        print(f"19_louisiana| flag mean: {ula_df.louisiana_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag
    _record('20_georgia_all', ula_df)
    if verbose:
        print(f"20_georgia  | flag mean: {ula_df.georgia_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag
    _record('21_txca_all', ula_df)
    if verbose:
        print(f"21_txca     | flag mean: {ula_df.txca_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'state_counter_adj':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag), 'loss_multiplier'] *= 1.012
    _record('22_state_counter_all', ula_df)
    if verbose:
        no_state_adj = ((~ula_df.louisiana_flag) & (~ula_df.georgia_flag) & (~ula_df.txca_flag)).mean()
        print(f"22_st_cntr  | no state adj: {no_state_adj:.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag
    _record('23_secured_credit_3.1+', ula_df)
    if verbose:
        print(f"23_sec_cr31 | chime mean: {ula_df.chime_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag
    _record('24_job_time_3.1+', ula_df)
    if verbose:
        print(f"24_job_t_31 | flag mean: {ula_df.job_time_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag
    _record('25_existing_dq_3.1+', ula_df)
    if verbose:
        print(f"25_dq_31    | flag mean: {ula_df.existing_dq_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    _record('26_employment_3.1+', ula_df)
    if verbose:
        print(f"26_emp_31   | seasonal: {ula_df.seasonal_employment_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag
    _record('27_auth_tradelines_3.1+', ula_df)
    if verbose:
        print(f"27_auth_31  | flag mean: {ula_df.kmx_auth_tradelines_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    mtn31_mask = ula_df.mtn_model.isin([3.1, 3.2, 4.1])
    sp_mask_31 = mtn31_mask & ula_df.soft_pull_flag
    hp_mask_31 = mtn31_mask & ~ula_df.soft_pull_flag

    if leave_out != 'Soft pull':
        ula_df.loc[sp_mask_31, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.low_bureau_flag
    _record('28a_sp_low_bureau_3.1+', ula_df)
    if verbose:
        sp_pop = ula_df[sp_mask_31]
        print(f"28a_sp_lb   | SP pct: {sp_mask_31.mean():.6f}  low_bureau(SP): {sp_pop.low_bureau_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[sp_mask_31, 'loss_multiplier'] *= 0.978 + 0.172 * ula_df.open_tl_flag
    _record('28b_sp_open_tl_3.1+', ula_df)
    if verbose:
        print(f"28b_sp_otl  | open_tl(SP): {sp_pop.open_tl_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[hp_mask_31, 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.low_bureau_flag
    _record('28c_hp_low_bureau_3.1+', ula_df)
    if verbose:
        hp_pop = ula_df[hp_mask_31]
        print(f"28c_hp_lb   | HP pct: {hp_mask_31.mean():.6f}  low_bureau(HP): {hp_pop.low_bureau_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[hp_mask_31, 'loss_multiplier'] *= 0.945 + 0.405 * ula_df.open_tl_flag
    _record('28d_hp_open_tl_3.1+', ula_df)
    if verbose:
        print(f"28d_hp_otl  | open_tl(HP): {hp_pop.open_tl_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[hp_mask_31, 'loss_multiplier'] /= np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1)
    _record('28e_hp_interaction_3.1+', ula_df)
    if verbose:
        hp_both = (hp_mask_31 & ula_df.cd_perc_flag.astype(bool) & ula_df.open_tl_flag.astype(bool)).mean()
        print(f"28e_hp_int  | HP cd_perc & open_tl pct: {hp_both:.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    sp_mask_312 = ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag
    hp_mask_312 = ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag
    sp_mask_41 = (ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag
    hp_mask_41 = (ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag

    if leave_out != 'Soft pull':
        ula_df.loc[sp_mask_312, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
    _record('28f_sp_cd_perc_3.1/3.2', ula_df)
    if verbose:
        sp_312 = ula_df[sp_mask_312]
        print(f"28f_sp_cdp  | SP 3.1/3.2 pct: {sp_mask_312.mean():.6f}  cd_perc(SP): {sp_312.cd_perc_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[hp_mask_312, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
    _record('28g_hp_cd_perc_3.1/3.2', ula_df)
    if verbose:
        hp_312 = ula_df[hp_mask_312]
        print(f"28g_hp_cdp  | HP 3.1/3.2 pct: {hp_mask_312.mean():.6f}  cd_perc(HP): {hp_312.cd_perc_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[sp_mask_41, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
    _record('28h_sp_cd_perc_4.1', ula_df)
    if verbose:
        sp_41 = ula_df[sp_mask_41]
        print(f"28h_sp_cdp  | SP 4.1 pct: {sp_mask_41.mean():.6f}  cd_perc(SP): {sp_41.cd_perc_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[hp_mask_41, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag
    _record('28i_hp_cd_perc_4.1', ula_df)
    if verbose:
        hp_41 = ula_df[hp_mask_41]
        print(f"28i_hp_cdp  | HP 4.1 pct: {hp_mask_41.mean():.6f}  cd_perc(HP): {hp_41.cd_perc_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'blanket adjustment':
        ula_df.loc[hp_mask_31, 'loss_multiplier'] /= 1.1
    _record('28j_blanket_hp_3.1+', ula_df)
    if verbose:
        print(f"28j_blanket | HP 3.1+ /= 1.1  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)
    _record('29_final_clip_3.1+', ula_df)
    if verbose:
        print(f"29_final_31 | clip [0.75, 1.4] for MTN 3.1+  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")
        print("\n--- Per MTN Model Means ---")
        for model in [3.0, 3.1, 3.2, 4.1]:
            mean_val = ula_df.loc[ula_df.mtn_model == model, 'loss_multiplier'].mean()
            count_val = (ula_df.mtn_model == model).sum()
            print(f"mtn_model {model}: mean loss_multiplier = {mean_val:.6f}  (n={count_val})")

    n = len(ula_df)
    flags = {
        'job_time_flag': ula_df.job_time_flag.mean(),
        'low_fico_flag': ula_df.low_fico_flag.mean(),
        'high_model_score_flag': ula_df.high_model_score_flag.mean(),
        'low_vantage_flag': ula_df.low_vantage_flag.mean(),
        'normal_pti_flag': ula_df.normal_pti_flag.mean(),
        'high_pti_tier_1_flag': ula_df.high_pti_tier_1_flag.mean(),
        'high_pti_tier_2_flag': ula_df.high_pti_tier_2_flag.mean(),
        'high_pti_tier_3_flag': ula_df.high_pti_tier_3_flag.mean(),
        'existing_dq_flag': ula_df.existing_dq_flag.mean(),
        'seasonal_employment_flag': ula_df.seasonal_employment_flag.mean(),
        'secured_credit_flag': ula_df.secured_credit_flag.mean(),
        'kmx_auth_tradelines_flag': ula_df.kmx_auth_tradelines_flag.mean(),
        'null_fico_w_vantage_flag': ula_df.null_fico_w_vantage_flag.mean(),
        'null_fico_null_vantage_flag': ula_df.null_fico_null_vantage_flag.mean(),
        'soft_pull_flag': ula_df.soft_pull_flag.mean(),
        'narrowed_soft_pull_flag': ula_df.narrowed_soft_pull_flag.mean(),
        'fraud_adjustment_mean': ula_df.fraud_adjustment.mean(),
        'continuous_vehicle_age_mean': ula_df.continuous_vehicle_age.mean(),
        'kmx_npc_flag': ula_df.kmx_npc_flag.mean(),
        'high_pti_npc': ula_df.high_pti_npc.mean(),
        'student_loan_flag': ula_df.student_loan_flag.mean(),
        'high_sales_price_flag': ula_df.high_sales_price_flag.mean(),
        'driver_flag': ula_df.driver_flag.mean(),
        'louisiana_flag': ula_df.louisiana_flag.mean(),
        'georgia_flag': ula_df.georgia_flag.mean(),
        'txca_flag': ula_df.txca_flag.mean(),
        'chime_flag': ula_df.chime_flag.mean(),
        'low_bureau_flag': ula_df.low_bureau_flag.mean(),
        'cd_perc_flag': ula_df.cd_perc_flag.mean(),
        'open_tl_flag': ula_df.open_tl_flag.mean(),
        'mtn_3_1_flag': ula_df.mtn_3_1_flag.mean(),
        'mtn_model_3.0_pct': (ula_df.mtn_model == 3.0).mean(),
        'mtn_model_3.1_pct': (ula_df.mtn_model == 3.1).mean(),
        'mtn_model_3.2_pct': (ula_df.mtn_model == 3.2).mean(),
        'mtn_model_4.1_pct': (ula_df.mtn_model == 4.1).mean(),
        'soft_pull_pct_3.1+': ula_df.loc[mtn31_mask, 'soft_pull_flag'].mean() if mtn31_mask.any() else float('nan'),
        'low_bureau_flag_sp': ula_df.loc[sp_mask_31, 'low_bureau_flag'].mean() if sp_mask_31.any() else float('nan'),
        'low_bureau_flag_hp': ula_df.loc[hp_mask_31, 'low_bureau_flag'].mean() if hp_mask_31.any() else float('nan'),
        'open_tl_flag_sp': ula_df.loc[sp_mask_31, 'open_tl_flag'].mean() if sp_mask_31.any() else float('nan'),
        'open_tl_flag_hp': ula_df.loc[hp_mask_31, 'open_tl_flag'].mean() if hp_mask_31.any() else float('nan'),
        'cd_perc_flag_sp': ula_df.loc[sp_mask_31, 'cd_perc_flag'].mean() if sp_mask_31.any() else float('nan'),
        'cd_perc_flag_hp': ula_df.loc[hp_mask_31, 'cd_perc_flag'].mean() if hp_mask_31.any() else float('nan'),
    }

    return ula_df, pd.Series(steps), pd.Series(flags)

In [5]:
mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config, leave_out='None'):
    """Core RAGU Score calculation for a single vintage and individual LOB."""
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']
    ltv_mult = 17 / 0.65 if lob == 'KMX' else 17
    apr_mult = 0.7 / 0.65 if lob == 'KMX' else 0.7

    ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob == lob)].copy()

    if len(ula_df) == 0:
        return None

    if lob == 'KMX':
        ula_df = get_ula_multiplier_kmx(ula_df, leave_out=leave_out)
    else:
        raise ValueError("This notebook only supports KMX")

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue', 'apr'],
        include_groups=False
    )

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()]
    recovery_df = recovery_df.copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics)

    vintage_ms_df = ms_df[ms_df['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * ltv_mult
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * apr_mult
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df

In [6]:
# Filter out Core LOB
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Assign period columns using pd.to_period
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')
    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter to configured date range
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# Convert week columns to str for compatibility
for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

# String version of date_col for flag comparisons
date_col_str = f'{date_col}_str'
ula_df_total[date_col_str] = ula_df_total[date_col].astype(str)

ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select([ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
                                      ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
                                     ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# Driver flag
warnings.filterwarnings('ignore', category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings('default', category=UserWarning)

# Handle NA values
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# Weekly-matching filters
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

# KMX Flags
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['null_fico_null_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & ((ula_df_total.vantage_score < 300) | (ula_df_total.vantage_score > 850))
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = (ula_df_total.cd_model_score >= 130) & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 475))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0

# Deduplicate driver flags
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# Vintage assignment
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# Filter to KMX only
ula_df_total = ula_df_total[ula_df_total.lob == 'KMX'].copy()
new_recovery = new_recovery[new_recovery.lob == 'KMX'].copy()

# MTN 4.1 model score transformation (applied in-place to ULA source)
is_mtn41 = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41, 'cd_model_score'] - 142) * 1.5
)

# Aggregate model scores from ULA data (same source as ltv/apr)
ms_df = ula_df_total.groupby(['period', 'lob']).apply(
    weighted_average_and_sum, 'cd_model_score', include_groups=False
).reset_index()
ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
ms_df['period'] = format_vintage(ms_df['period'])

print(f'Data loaded: {len(ula_df_total)} ULA rows')
print(f'MTN model distribution:\n{ula_df_total.mtn_model.value_counts().sort_index()}')
print(f'ms_df: {len(ms_df)} rows')
print(f'Available KMX vintages: {sorted(ula_df_total.vintage.unique())}')

Data loaded: 222183 ULA rows
MTN model distribution:
mtn_model
3.0    132720
3.1     51883
3.2     29746
4.1      7834
Name: count, dtype: int64
ms_df: 15 rows
Available KMX vintages: ['2023 Q1', '2023 Q2', '2023 Q3', '2023 Q4', '2024 Q1', '2024 Q2', '2024 Q3', '2024 Q4', '2025 Q1', '2025 Q2', '2025 Q3', '2025 Q4', '2026 Q1', '2026 Q2', '2026 Q3']


In [7]:
baseline_config = BASELINES['KMX']

results_by_model = {}

all_vintages = sorted(ula_df_total['vintage'].unique())

for mtn_model_filter in MTN_MODELS + ['All KMX']:
    label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
    print(f'\n{"="*60}')
    print(f'  Processing: {label}')
    print(f'{"="*60}')

    # Filter DataFrames by MTN model
    if mtn_model_filter == 'All KMX':
        ula_filtered = ula_df_total.copy()
        new_rec_filtered = new_recovery.copy()
        ms_filtered = ms_df.copy()
    else:
        mtn_accounts = set(ula_df_total[ula_df_total.mtn_model == mtn_model_filter].account_number)
        ula_filtered = ula_df_total[ula_df_total.mtn_model == mtn_model_filter].copy()
        new_rec_filtered = new_recovery[new_recovery.account_number.isin(mtn_accounts)].copy()
        ms_filtered = rebuild_ms_df(ula_filtered)

    n_loans = len(ula_filtered)
    if n_loans == 0:
        print(f'  No loans found for {label}, skipping.')
        results_by_model[mtn_model_filter] = pd.DataFrame()
        continue

    print(f'  Loans: {n_loans}')

    full_df_list = []
    vintages_processed = []

    for vintage in all_vintages:
        n_vintage = len(ula_filtered[ula_filtered.vintage == vintage])
        if n_vintage == 0:
            continue

        print(f'  {vintage} KMX ({label}, n={n_vintage})')
        full_df = get_ragu_score(vintage, 'KMX', ula_filtered, new_rec_filtered, ms_filtered, baseline_config)
        if full_df is not None:
            full_df_list.append(full_df)
            vintages_processed.append(vintage)

    if full_df_list:
        all_df = pd.concat(full_df_list)
        results_by_model[mtn_model_filter] = all_df
        print(f'\n  {label}: {len(vintages_processed)} vintages processed')
    else:
        results_by_model[mtn_model_filter] = pd.DataFrame()
        print(f'\n  {label}: No vintages had data')

print(f'\n{"="*60}')
print('  All models processed.')
print(f'{"="*60}')
print("[PROGRESS] Scoring Complete")



  Processing: MTN 3.0
  Loans: 132720
  2023 Q1 KMX (MTN 3.0, n=14355)
  2023 Q2 KMX (MTN 3.0, n=12711)
  2023 Q3 KMX (MTN 3.0, n=12892)
  2023 Q4 KMX (MTN 3.0, n=10521)
  2024 Q1 KMX (MTN 3.0, n=18128)
  2024 Q2 KMX (MTN 3.0, n=13833)
  2024 Q3 KMX (MTN 3.0, n=12991)
  2024 Q4 KMX (MTN 3.0, n=8929)
  2025 Q1 KMX (MTN 3.0, n=8813)
  2025 Q2 KMX (MTN 3.0, n=7452)
  2025 Q3 KMX (MTN 3.0, n=7250)
  2025 Q4 KMX (MTN 3.0, n=4799)
  2026 Q3 KMX (MTN 3.0, n=46)

  MTN 3.0: 13 vintages processed

  Processing: MTN 3.1
  Loans: 51883
  2024 Q4 KMX (MTN 3.1, n=3231)
  2025 Q1 KMX (MTN 3.1, n=9947)
  2025 Q2 KMX (MTN 3.1, n=8172)
  2025 Q3 KMX (MTN 3.1, n=8190)
  2025 Q4 KMX (MTN 3.1, n=8612)
  2026 Q1 KMX (MTN 3.1, n=13018)
  2026 Q2 KMX (MTN 3.1, n=713)

  MTN 3.1: 7 vintages processed

  Processing: MTN 3.2
  Loans: 29746
  2026 Q1 KMX (MTN 3.2, n=7451)
  2026 Q2 KMX (MTN 3.2, n=14603)
  2026 Q3 KMX (MTN 3.2, n=7692)

  MTN 3.2: 3 vintages processed

  Processing: MTN 4.1
  Loans: 7834
  2025

In [8]:
"""
Diagnostics: Run get_ula_multiplier_kmx_diag for each MTN model and for All KMX combined.
Produces step-by-step loss_multiplier traces and flag means per vintage.
"""

def build_output_df(results, wtd_mults, record_counts, ragu_gli_dict=None):
    """Helper to build multiplier steps and summary DataFrame."""
    if not results:
        return None
    step_names = list(next(iter(results.values())).keys())
    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        col_data[col_name] = [step_dict.get(s) for s in step_names]
    diag_df = pd.DataFrame(col_data, index=step_names)
    summary = {}
    for (lob, vintage) in results.keys():
        col = f"{lob} | {vintage}"
        final_mult_mean = diag_df[col].iloc[-1]
        wtd_mult = wtd_mults.get((lob, vintage), float('nan'))
        if ragu_gli_dict is not None:
            gross_loss_impact = ragu_gli_dict.get((lob, vintage), 25 * (1 - wtd_mult))
        else:
            gross_loss_impact = 25 * (1 - wtd_mult)
        summary[col] = {
            '--- FINAL_MULT (mean)': final_mult_mean,
            '--- WTD_MULT_RAGU': wtd_mult,
            '--- GROSS_LOSS_IMPACT': gross_loss_impact,
            '--- N_RECORDS': record_counts.get((lob, vintage), 0),
        }
    summary_df = pd.DataFrame(summary)
    return pd.concat([diag_df, summary_df])

def build_flags_df(flag_results, record_counts):
    """Helper to build flag means DataFrame."""
    if not flag_results:
        return None
    flag_names = list(next(iter(flag_results.values())).keys())
    flag_col_data = {}
    for (lob, vintage), flag_dict in flag_results.items():
        col_name = f"{lob} | {vintage}"
        flag_col_data[col_name] = [flag_dict.get(f) for f in flag_names]
    flags_df = pd.DataFrame(flag_col_data, index=flag_names)
    n_row = {}
    for (lob, vintage) in flag_results.keys():
        n_row[f"{lob} | {vintage}"] = record_counts.get((lob, vintage), 0)
    flags_df.loc['--- N_RECORDS'] = n_row
    return flags_df


STEP_LABEL_MAP = {
    '00_initial':              'Initial (1.0)',
    '02_low_fico_3.0':         'Low FICO (3.0)',
    '03_low_vantage_3.0':      'Low Vantage (3.0)',
    '04_high_pti_3.0':         'High PTI (3.0)',
    '07_loss_scale_div_3.0':   'Loss Scale Div (3.0)',
    '08_secured_credit_3.0':   'Secured Credit (3.0)',
    '09_auth_tradelines_3.0':  'Auth Tradelines (3.0)',
    '11_soft_pull_3.0':        'Soft Pull (3.0)',
    '12_fraud_all':            'Fraud Adjustment',
    '13_clip_3.0':             'Clip (3.0)',
    '14_vehicle_age_3.0':      'Vehicle Age (3.0)',
    '15_npc_all':              'NPC',
    '16_student_loans_all':    'Student Loans',
    '17_high_sales_price_4.1': 'High Sales Price (4.1)',
    '18_driver_flag_all':      'Driver Flag',
    '19_louisiana_all':        'Louisiana',
    '20_georgia_all':          'Georgia',
    '21_txca_all':             'TX / CA',
    '22_state_counter_all':    'State Counter',
    '23_secured_credit_3.1+':  'Secured Credit / Chime (3.1+)',
    '24_job_time_3.1+':        'Job Time (3.1+)',
    '25_existing_dq_3.1+':     'Existing DQ (3.1+)',
    '26_employment_3.1+':      'Employment Type (3.1+)',
    '27_auth_tradelines_3.1+': 'Auth Tradelines (3.1+)',
    '28a_sp_low_bureau_3.1+':   'Low Bureau (SP, 3.1+)',
    '28b_sp_open_tl_3.1+':     'Open TL (SP, 3.1+)',
    '28c_hp_low_bureau_3.1+':  'Low Bureau (HP, 3.1+)',
    '28d_hp_open_tl_3.1+':     'Open TL (HP, 3.1+)',
    '28e_hp_interaction_3.1+': 'HP Interaction (3.1+)',
    '28f_sp_cd_perc_3.1/3.2':  'CD Perc (SP, 3.1/3.2)',
    '28g_hp_cd_perc_3.1/3.2':  'CD Perc (HP, 3.1/3.2)',
    '28h_sp_cd_perc_4.1':      'CD Perc (SP, 4.1)',
    '28i_hp_cd_perc_4.1':      'CD Perc (HP, 4.1)',
    '28j_blanket_hp_3.1+':     'Blanket Adjustment (HP, 3.1+)',
    '29_final_clip_3.1+':      'Clip (3.1+)',
}


def build_attribution_df(results, ragu_gli_dict):
    """
    Decompose gross_loss_impact across multiplier steps using logarithmic attribution.

    For each vintage:
      1. Compute per-step ratios: r_i = step_i / step_{i-1}
      2. Log shares: ln(r_i) / ln(final_multiplier)  [proportional contribution]
      3. Derive equal-weighted GLI from the diagnostic chain: 25 * (1 - final_mult)
      4. Attributed impact: share_i * diag_gli
      5. Population Weighting row: ragu_gli - subtotal (bridges equal-weighted
         to the amount-financed-weighted RAGU GLI)

    Step attributions use the diagnostic chain's own final multiplier so that
    signs are always consistent: reducing the multiplier yields positive
    attribution (lower mult -> higher GLI -> higher RAGU score).

    The Population Weighting row captures the effect of dollar-weighting and
    bb-population filtering.  The TOTAL row matches the RAGU GLI exactly.
    """
    if not results:
        return None

    step_keys = list(next(iter(results.values())).keys())
    adjustment_keys = [k for k in step_keys if k != '00_initial']
    readable_labels = [STEP_LABEL_MAP.get(k, k) for k in adjustment_keys]

    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        ragu_gli = ragu_gli_dict.get((lob, vintage), float('nan'))

        cumulative = [step_dict.get(k, float('nan')) for k in step_keys]
        ratios = []
        for i, k in enumerate(step_keys):
            if k == '00_initial':
                continue
            prev = cumulative[i - 1]
            curr = cumulative[i]
            if prev and prev != 0:
                ratios.append(curr / prev)
            else:
                ratios.append(1.0)

        import math
        final_mult = cumulative[-1]
        log_final = math.log(final_mult) if final_mult and final_mult > 0 and abs(final_mult - 1.0) > 1e-12 else None
        diag_gli = 25.0 * (1.0 - final_mult) if final_mult and not pd.isna(final_mult) else 0.0

        ragu_gli_val = ragu_gli if not pd.isna(ragu_gli) else 0.0
        if log_final is None:
            weighting_adj = ragu_gli_val - diag_gli
            col_data[col_name] = [0.0] * len(adjustment_keys) + [weighting_adj, ragu_gli_val]
        else:
            log_ratios = [math.log(r) if r and r > 0 else 0.0 for r in ratios]
            attributed = [(lr / log_final) * diag_gli for lr in log_ratios]
            weighting_adj = ragu_gli_val - sum(attributed)
            col_data[col_name] = attributed + [weighting_adj, ragu_gli_val]

    index_labels = readable_labels + ['--- Population Weighting', '--- TOTAL (RAGU GLI)']
    return pd.DataFrame(col_data, index=index_labels)

diag_results_by_model = {}

for mtn_model_filter in MTN_MODELS + ['All KMX']:
    label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
    
    if mtn_model_filter == 'All KMX':
        ula_diag_source = ula_df_total.copy()
    else:
        ula_diag_source = ula_df_total[ula_df_total.mtn_model == mtn_model_filter].copy()
    
    if len(ula_diag_source) == 0:
        print(f'\n{label}: No data for diagnostics, skipping.')
        diag_results_by_model[mtn_model_filter] = (None, None, None)
        continue
    
    # Build ragu_gli_dict from results_by_model for this MTN model
    ragu_gli_dict = {}
    model_df = results_by_model.get(mtn_model_filter)
    if model_df is not None and len(model_df) > 0:
        for _, row in model_df.reset_index().iterrows():
            ragu_gli_dict[('KMX', row['vintage'])] = row['gross_loss_impact']
    
    # Get all vintages for diagnostics
    target_vintages = sorted(ula_diag_source.vintage.unique())
    
    kmx_results = {}
    kmx_flag_results = {}
    kmx_wtd_mults = {}
    kmx_record_counts = {}
    
    print(f'\n{"="*60}')
    print(f'  DIAGNOSTICS: {label}')
    print(f'{"="*60}')
    
    for vintage in target_vintages:
        ula_vintage = ula_diag_source[ula_diag_source.vintage == vintage].copy()
        n = len(ula_vintage)
        if n == 0:
            continue
        
        print(f'\n{"="*60}')
        print(f'  KMX {vintage} ({label})  (n={n})')
        print(f'{"="*60}')
        
        ula_vintage_diag, steps, flags = get_ula_multiplier_kmx_diag(ula_vintage, loss_scale=MODEL_PARAMS['kmx_loss_scale'], leave_out='None', verbose=True)
        
        diag_mix = ula_vintage_diag[['account_number', 'bbvalue', 'amt_financed', 'loss_multiplier']].copy()
        nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
            subset='account_number', keep='first')
        diag_mix = diag_mix.merge(nr, on='account_number', how='left').drop_duplicates(
            subset='account_number', keep='first')
        bb_pop = diag_mix[diag_mix['bbvalue'].notna() & (diag_mix['bbvalue'] > 0)]
        if len(bb_pop) > 0 and bb_pop.amt_financed.sum() > 0:
            wtd_mult = (bb_pop.loss_multiplier * bb_pop.amt_financed).sum() / bb_pop.amt_financed.sum()
        else:
            wtd_mult = float('nan')
        
        ragu_gli = ragu_gli_dict.get(('KMX', vintage), float('nan'))
        print(f"  bb_populated: {len(bb_pop)} / {n}  wtd_mult: {wtd_mult:.6f}  ragu_gli: {ragu_gli:.4f}")
        
        kmx_results[('KMX', vintage)] = steps.to_dict()
        kmx_flag_results[('KMX', vintage)] = flags.to_dict()
        kmx_wtd_mults[('KMX', vintage)] = wtd_mult
        kmx_record_counts[('KMX', vintage)] = n
    
    kmx_output_df = build_output_df(kmx_results, kmx_wtd_mults, kmx_record_counts, ragu_gli_dict)
    kmx_flags_df = build_flags_df(kmx_flag_results, kmx_record_counts)
    kmx_attribution_df = build_attribution_df(kmx_results, ragu_gli_dict)
    diag_results_by_model[mtn_model_filter] = (kmx_output_df, kmx_flags_df, kmx_attribution_df)

    if kmx_output_df is not None:
        print(f'\n--- {label} Multiplier Steps ---')
        display(kmx_output_df)
        print(f'\n--- {label} Flag Means ---')
        display(kmx_flags_df)
    if kmx_attribution_df is not None:
        print(f'\n--- {label} Gross Loss Attribution ---')
        display(kmx_attribution_df)


  DIAGNOSTICS: MTN 3.0

  KMX 2023 Q1 (MTN 3.0)  (n=14355)
00_initial  | loss_multiplier mean: 1.000000
02_low_fico | flag mean: 0.082132  | loss_multiplier mean: 1.019539
03_low_vant | flag mean: 0.000000  | loss_multiplier mean: 1.019539
04_high_pti | tier1: 0.099060  tier2: 0.047092  tier3: 0.003622  | loss_multiplier mean: 1.031056
07_scale_dv | dividing by (1 + 0.067)  | loss_multiplier mean: 0.966313
08_sec_cred | flag mean: 0.182375  | loss_multiplier mean: 0.966580
09_auth_tl  | flag mean: 0.039429  | loss_multiplier mean: 0.963798
11_soft_pll | flag mean: 0.635040  narrowed: 0.099060  | loss_multiplier mean: 0.940376
12_fraud    | fraud_adj mean: 1.003439  | loss_multiplier mean: 0.943189
13_clip_3.0 | clip [0.8, 1.2652]  | loss_multiplier mean: 0.941898
14_veh_age  | continuous_age mean: 6.565430  | loss_multiplier mean: 0.934088
15_npc      | npc_flag mean: 0.641519  high_pti_npc mean: 0.149774  | loss_multiplier mean: 0.938013
16_stu_loan | flag mean: 0.042076  | loss_mult

,KMX | 2023 Q1,KMX | 2023 Q2,KMX | 2023 Q3,KMX | 2023 Q4,KMX | 2024 Q1,KMX | 2024 Q2,KMX | 2024 Q3,KMX | 2024 Q4,KMX | 2025 Q1,KMX | 2025 Q2,KMX | 2025 Q3,KMX | 2025 Q4,KMX | 2026 Q3
00_initial,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
02_low_fico_3.0,1.019539,1.014565,1.012375,1.010877,1.008490,1.006261,1.005939,1.005517,1.000263,1.000000,1.000000,1.000000,1.000000
03_low_vantage_3.0,1.019539,1.014565,1.012375,1.010877,1.008490,1.006261,1.005939,1.005524,1.001902,1.002054,1.001778,1.001793,1.000000
04_high_pti_3.0,1.031056,1.026466,1.025155,1.024821,1.018635,1.017855,1.017173,1.016710,1.012209,1.010900,1.010491,1.012196,1.006522
07_loss_scale_div_3.0,0.966313,0.962011,0.960783,0.960470,0.954672,0.953941,0.953302,0.952868,0.948650,0.947423,0.947039,0.948637,0.943319
08_secured_credit_3.0,0.966580,0.965348,0.963220,0.962390,0.959284,0.958631,0.958222,0.958243,0.961675,0.954237,0.955275,0.957093,0.935109
09_auth_tradelines_3.0,0.963798,0.963671,0.962134,0.960322,0.955397,0.955863,0.955562,0.954559,0.957559,0.950509,0.951328,0.954659,0.925758
11_soft_pull_3.0,0.940376,0.941918,0.939997,0.937221,0.931381,0.931932,0.931227,0.930319,0.934869,0.926877,0.928611,0.931218,0.902658
12_fraud_all,0.943189,0.945137,0.943049,0.940616,0.924281,0.922186,0.926978,0.928134,0.935783,0.926924,0.928073,0.936525,0.906270
13_clip_3.0,0.941898,0.943470,0.941399,0.938917,0.923856,0.921131,0.925740,0.925953,0.934176,0.926259,0.927291,0.935196,0.906270



--- MTN 3.0 Flag Means ---


,KMX | 2023 Q1,KMX | 2023 Q2,KMX | 2023 Q3,KMX | 2023 Q4,KMX | 2024 Q1,KMX | 2024 Q2,KMX | 2024 Q3,KMX | 2024 Q4,KMX | 2025 Q1,KMX | 2025 Q2,KMX | 2025 Q3,KMX | 2025 Q4,KMX | 2026 Q3
job_time_flag,0.246883,0.227519,0.235805,0.224693,0.186176,0.163233,0.181741,0.162728,0.128560,0.128288,0.146483,0.181079,0.217391
low_fico_flag,0.082132,0.061600,0.052280,0.048379,0.039331,0.032242,0.029559,0.027887,0.001135,0.000000,0.000000,0.000000,0.000000
high_model_score_flag,0.221665,0.241916,0.234021,0.241422,0.197981,0.246873,0.230929,0.222869,0.218200,0.231616,0.222621,0.209002,0.543478
low_vantage_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000112,0.008964,0.010870,0.008828,0.008543,0.000000
normal_pti_flag,0.850226,0.848556,0.837263,0.836137,0.870256,0.865105,0.870372,0.878934,0.873028,0.882448,0.884138,0.876016,0.891304
high_pti_tier_1_flag,0.099060,0.104319,0.111154,0.101796,0.081973,0.081544,0.077900,0.071117,0.083626,0.076355,0.073793,0.076058,0.086957
high_pti_tier_2_flag,0.047092,0.042247,0.046385,0.055508,0.044572,0.047857,0.046263,0.043342,0.038806,0.038916,0.040000,0.043342,0.021739
high_pti_tier_3_flag,0.003622,0.004878,0.005197,0.006558,0.003199,0.005494,0.005465,0.006608,0.004539,0.002281,0.002069,0.004584,0.000000
existing_dq_flag,0.084291,0.073716,0.068647,0.070145,0.081311,0.080026,0.073282,0.070669,0.077726,0.081991,0.081793,0.081059,0.000000
seasonal_employment_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.006581,0.007246,0.007724,0.009377,0.000000



--- MTN 3.0 Gross Loss Attribution ---


...

,KMX | 2023 Q1,KMX | 2023 Q2,KMX | 2023 Q3,KMX | 2023 Q4,KMX | 2024 Q1,KMX | 2024 Q2,KMX | 2024 Q3,KMX | 2024 Q4,KMX | 2025 Q1,KMX | 2025 Q2,KMX | 2025 Q3,KMX | 2025 Q4,KMX | 2026 Q1,KMX | 2026 Q2,KMX | 2026 Q3
00_initial,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
02_low_fico_3.0,1.019539,1.014565,1.012375,1.010877,1.008490,1.006261,1.005939,1.004051,1.000124,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
03_low_vantage_3.0,1.019539,1.014565,1.012375,1.010877,1.008490,1.006261,1.005939,1.004056,1.000893,1.000980,1.000835,1.000631,1.000000,1.000000,1.000000
04_high_pti_3.0,1.031056,1.026466,1.025155,1.024821,1.018635,1.017855,1.017173,1.012270,1.005736,1.005199,1.004926,1.004294,1.000000,1.000000,1.000031
07_loss_scale_div_3.0,0.966313,0.962011,0.960783,0.960470,0.954672,0.953941,0.953302,0.965391,0.975877,0.974923,0.975132,0.981914,1.000000,1.000000,0.999734
08_secured_credit_3.0,0.966580,0.965348,0.963220,0.962390,0.959284,0.958631,0.958222,0.969338,0.981996,0.978173,0.978999,0.984892,1.000000,1.000000,0.999696
09_auth_tradelines_3.0,0.963798,0.963671,0.962134,0.960322,0.955397,0.955863,0.955562,0.966633,0.980062,0.976395,0.977146,0.984035,1.000000,1.000000,0.999652
11_soft_pull_3.0,0.940376,0.941918,0.939997,0.937221,0.931381,0.931932,0.931227,0.948834,0.969403,0.965123,0.966479,0.975781,1.000000,1.000000,0.999543
12_fraud_all,0.943189,0.945137,0.943049,0.940616,0.924281,0.922186,0.926978,0.945007,0.964628,0.959070,0.961657,0.975361,0.997482,1.016433,1.012998
13_clip_3.0,0.941898,0.943470,0.941399,0.938917,0.923856,0.921131,0.925740,0.943406,0.963873,0.958753,0.961289,0.974893,0.997482,1.016433,1.012998



--- All KMX Flag Means ---


,KMX | 2023 Q1,KMX | 2023 Q2,KMX | 2023 Q3,KMX | 2023 Q4,KMX | 2024 Q1,KMX | 2024 Q2,KMX | 2024 Q3,KMX | 2024 Q4,KMX | 2025 Q1,KMX | 2025 Q2,KMX | 2025 Q3,KMX | 2025 Q4,KMX | 2026 Q1,KMX | 2026 Q2,KMX | 2026 Q3
job_time_flag,0.246883,0.227519,0.235805,0.224693,0.186176,0.163233,0.181741,0.178454,0.162154,0.116999,0.126101,0.142197,0.113215,0.111511,0.120922
low_fico_flag,0.082132,0.061600,0.052280,0.048379,0.039331,0.032242,0.029559,0.039638,0.001386,0.000000,0.000000,0.000000,0.000261,0.000000,0.000000
high_model_score_flag,0.221665,0.241916,0.234021,0.241422,0.197981,0.246873,0.230929,0.241283,0.237793,0.267857,0.259650,0.265904,0.304213,0.328812,0.355730
low_vantage_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000740,0.014712,0.013313,0.010492,0.011740,0.016845,0.012257,0.008768
normal_pti_flag,0.850226,0.848556,0.837263,0.836137,0.870256,0.865105,0.870372,0.869326,0.860075,0.872440,0.878174,0.876146,0.900888,0.896497,0.892129
high_pti_tier_1_flag,0.099060,0.104319,0.111154,0.101796,0.081973,0.081544,0.077900,0.074342,0.084382,0.077573,0.073705,0.075648,0.065465,0.069183,0.068210
high_pti_tier_2_flag,0.047092,0.042247,0.046385,0.055508,0.044572,0.047857,0.046263,0.049013,0.049467,0.047043,0.044883,0.044024,0.030905,0.031977,0.036603
high_pti_tier_3_flag,0.003622,0.004878,0.005197,0.006558,0.003199,0.005494,0.005465,0.007319,0.006077,0.002944,0.003238,0.004182,0.002742,0.002342,0.003059
existing_dq_flag,0.084291,0.073716,0.068647,0.070145,0.081311,0.080026,0.073282,0.094984,0.130650,0.101639,0.101878,0.114095,0.135109,0.136515,0.128365
seasonal_employment_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008209,0.006848,0.007383,0.007924,0.007400,0.008553,0.007953



--- All KMX Gross Loss Attribution ---


,KMX | 2023 Q1,KMX | 2023 Q2,KMX | 2023 Q3,KMX | 2023 Q4,KMX | 2024 Q1,KMX | 2024 Q2,KMX | 2024 Q3,KMX | 2024 Q4,KMX | 2025 Q1,KMX | 2025 Q2,KMX | 2025 Q3,KMX | 2025 Q4,KMX | 2026 Q1,KMX | 2026 Q2,KMX | 2026 Q3
Low FICO (3.0),-0.477993,-0.356165,-0.301673,-0.263546,-0.208435,-0.151095,-0.144388,-0.099581,-0.003084,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
Low Vantage (3.0),-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000135,-0.019213,-0.023817,-0.020516,-0.015529,-0.000000,-0.000000,-0.000000
High PTI (3.0),-0.277491,-0.287250,-0.307722,-0.333731,-0.246767,-0.277318,-0.270816,-0.200698,-0.120504,-0.102290,-0.100302,-0.089926,-0.000000,-0.000000,-0.000764
Loss Scale Div (3.0),1.601970,1.597352,1.590727,1.579806,1.598839,1.569857,1.581375,1.168038,0.752507,0.743742,0.739976,0.554594,-0.000000,-0.000000,0.007410
Secured Credit (3.0),-0.006812,-0.085275,-0.062147,-0.048660,-0.118825,-0.118722,-0.125535,-0.100496,-0.156071,-0.080940,-0.097308,-0.074504,-0.000000,-0.000000,0.000963
Auth Tradelines (3.0),0.071188,0.042821,0.027663,0.052412,0.100122,0.069994,0.067795,0.068840,0.049208,0.044249,0.046582,0.021419,-0.000000,-0.000000,0.001097
Soft Pull (3.0),0.607725,0.562381,0.570964,0.593155,0.627634,0.613782,0.629037,0.457807,0.273056,0.282382,0.269875,0.207287,-0.000000,-0.000000,0.002709
Fraud Adjustment,-0.073788,-0.084049,-0.079508,-0.088069,0.188681,0.254483,0.111511,0.099539,0.123278,0.153000,0.122976,0.010592,0.063224,-0.409769,-0.334203
Clip (3.0),0.033836,0.043482,0.042960,0.044036,0.011338,0.027696,0.032586,0.041784,0.019548,0.008040,0.009393,0.011806,-0.000000,-0.000000,-0.000000
Vehicle Age (3.0),0.205696,0.217430,0.210691,0.404903,0.423132,0.851541,0.761230,0.607612,0.413396,0.361853,0.309271,0.230416,-0.000000,-0.000000,0.007638


In [9]:
def build_ragu_decomposition(all_df_model, original_model_scores_df, lob='KMX'):
    '''Build RAGU decomposition table from model results.'''
    if all_df_model is None or len(all_df_model) == 0:
        return None

    raw = all_df_model.loc[[lob]][['vintage', 'ms_original', 'gross_loss_impact', 'recovery_impact',
                                   'ltv_impact', 'apr_impact', 'ragu_score', 'ltv']].reset_index(drop=True).set_index('vintage').T.copy()

    ms_for_lob = original_model_scores_df[original_model_scores_df.lob == lob].copy()
    ms_for_lob = ms_for_lob[ms_for_lob['period'].isin(raw.columns)]
    if len(ms_for_lob) > 0:
        raw = pd.concat([raw, ms_for_lob.drop(columns=['lob', 'amt_financed']).rename(columns={'period': 'vintage'}).set_index('vintage').T.rename({'model_score': 'contract_model_score'})])

    raw.index = ['Mountain3 Score', 'Gross Loss Impact', 'Recovery Impact',
                 'LTV Impact', 'APR Impact', 'RAGU Score', 'LTV', 'Contract Model Score']

    decomp = pd.DataFrame(index=['Contract Model Score', 'Mountain3 Score',
                                  'Gross Loss Adjustments', 'Recovery Adjustments',
                                  'LTV Adjustments', 'APR Adjustments',
                                  'RAGU Score', 'LTV'],
                           columns=raw.columns)

    decomp.loc['Contract Model Score'] = raw.loc['Contract Model Score']
    decomp.loc['Mountain3 Score'] = raw.loc['Mountain3 Score'] - raw.loc['Contract Model Score']
    decomp.loc['Gross Loss Adjustments'] = raw.loc['Gross Loss Impact']
    decomp.loc['Recovery Adjustments'] = raw.loc['Recovery Impact']
    decomp.loc['LTV Adjustments'] = raw.loc['LTV Impact']
    decomp.loc['APR Adjustments'] = raw.loc['APR Impact']
    decomp.loc['RAGU Score'] = raw.loc['RAGU Score']
    decomp.loc['LTV'] = raw.loc['LTV']

    return decomp

# Build original_model_scores for each MTN model subset
original_model_scores_by_model = {}
for mtn_model_filter in MTN_MODELS + ['All KMX']:
    if mtn_model_filter == 'All KMX':
        original_model_scores_by_model['All KMX'] = ms_df
    else:
        ula_subset = ula_df_total[ula_df_total.mtn_model == mtn_model_filter]
        if len(ula_subset) > 0:
            original_model_scores_by_model[mtn_model_filter] = rebuild_ms_df(ula_subset)
        else:
            original_model_scores_by_model[mtn_model_filter] = pd.DataFrame(columns=['period', 'lob', 'model_score', 'amt_financed'])

# Write to Excel
with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    for mtn_model_filter in ['All KMX'] + MTN_MODELS:
        label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
        all_df_model = results_by_model.get(mtn_model_filter)
        orig_ms = original_model_scores_by_model.get(mtn_model_filter)

        if all_df_model is None or len(all_df_model) == 0:
            print(f'{label}: No results to export, writing empty sheet.')
            pd.DataFrame({'Note': [f'No loans found for {label}']}).to_excel(writer, sheet_name=label, index=False)
            continue

        xlsx_df = build_ragu_decomposition(all_df_model, orig_ms, lob='KMX')
        if xlsx_df is not None:
            xlsx_df.to_excel(writer, sheet_name=label)
            print(f'\n{label} RAGU Decomposition:')
            display(xlsx_df)
        else:
            pd.DataFrame({'Note': [f'Could not build decomposition for {label}']}).to_excel(writer, sheet_name=label, index=False)

    # Diagnostics sheets
    for mtn_model_filter in ['All KMX'] + MTN_MODELS:
        label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
        diag_data = diag_results_by_model.get(mtn_model_filter, (None, None, None))
        output_df, flags_df, attribution_df = diag_data

        sheet_mult = f'{label} Mult Steps'[:31]
        sheet_flags = f'{label} Flags'[:31]
        sheet_attr = f'{label} Attribution'[:31]

        if output_df is not None:
            output_df.to_excel(writer, sheet_name=sheet_mult)
        if flags_df is not None:
            flags_df.to_excel(writer, sheet_name=sheet_flags)
        if attribution_df is not None:
            attribution_df.to_excel(writer, sheet_name=sheet_attr)

print(f'\nExported to: {EXCEL_OUTPUT}')
print("[PROGRESS] Export Complete")



All KMX RAGU Decomposition:


vintage,2023 Q1,2023 Q2,2023 Q3,2023 Q4,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2,2026 Q3
Contract Model Score,140.103626,140.658174,140.490951,140.569789,139.563853,140.439796,139.879652,139.789069,139.725861,139.866194,139.908574,140.934457,142.167407,142.946899,143.525531
Mountain3 Score,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gross Loss Adjustments,1.077815,1.29163,1.459354,1.722564,1.260756,2.177068,1.87728,1.286326,0.540014,1.785166,1.239374,1.243884,0.181344,0.11529,0.146563
Recovery Adjustments,-0.873258,-0.258316,-0.302404,0.30539,-0.058021,0.279748,-0.358544,-0.489204,-0.540677,0.129312,0.64011,0.953646,0.819743,0.938954,0.760866
LTV Adjustments,-2.674912,-1.396738,-3.69043,-4.7719,-4.080808,-0.950288,-1.55379,-1.482701,-0.727484,-0.313402,-0.53154,-0.229127,0.476485,1.077952,1.345625
APR Adjustments,-0.246162,-0.167757,0.002127,-0.003132,-0.197908,0.335081,0.211894,0.25139,-0.087742,-0.054187,0.032568,0.696048,0.476706,0.698584,0.963382
RAGU Score,137.387109,140.126993,137.959597,137.822711,136.487872,142.281406,140.056493,139.354879,138.909971,141.413084,141.289086,143.598908,144.121685,145.777679,146.741967
LTV,1.771146,1.679704,1.851215,1.944847,1.883955,1.64995,1.690428,1.685557,1.635492,1.609284,1.622985,1.604053,1.561551,1.527061,1.512197



MTN 3.0 RAGU Decomposition:


vintage,2023 Q1,2023 Q2,2023 Q3,2023 Q4,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q3
Contract Model Score,140.103626,140.658174,140.490951,140.569789,139.563853,140.439796,139.879652,139.141183,138.624949,138.351644,138.576206,138.920424,145.598432
Mountain3 Score,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gross Loss Adjustments,1.077815,1.29163,1.459354,1.722564,1.260756,2.177068,1.87728,1.762248,1.009167,2.483169,1.863475,1.774811,5.474959
Recovery Adjustments,-0.873258,-0.258316,-0.302404,0.30539,-0.058021,0.279748,-0.358544,-0.524075,-0.832419,-0.170839,0.010354,0.292765,2.999967
LTV Adjustments,-2.674912,-1.396738,-3.69043,-4.7719,-4.080808,-0.950288,-1.55379,-1.60455,-1.195346,-0.878184,-1.394356,-1.457236,3.76413
APR Adjustments,-0.246162,-0.167757,0.002127,-0.003132,-0.197908,0.335081,0.211894,0.337147,-0.247797,-0.218601,-0.050674,0.794204,1.887093
RAGU Score,137.387109,140.126993,137.959597,137.822711,136.487872,142.281406,140.056493,139.111952,137.358554,139.567189,139.005005,140.324967,159.724581
LTV,1.771146,1.679704,1.851215,1.944847,1.883955,1.64995,1.690428,1.693923,1.66615,1.645243,1.679542,1.683819,1.389954



MTN 3.1 RAGU Decomposition:


...


MTN 3.2 RAGU Decomposition:


vintage,2026 Q1,2026 Q2,2026 Q3
Contract Model Score,142.080076,142.346376,142.909828
Mountain3 Score,0.0,0.0,0.0
Gross Loss Adjustments,0.02608,0.306795,0.397136
Recovery Adjustments,0.658454,0.991434,0.780448
LTV Adjustments,0.161948,1.001182,1.424252
APR Adjustments,0.225608,0.542783,0.828788
RAGU Score,143.152166,145.18857,146.340452
LTV,1.580215,1.531378,1.507886



MTN 4.1 RAGU Decomposition:


vintage,2025 Q4,2026 Q1,2026 Q2,2026 Q3
Contract Model Score,142.786061,144.262919,145.809776,145.794362
Mountain3 Score,0.0,0.0,0.0,0.0
Gross Loss Adjustments,-0.155561,-0.817758,-0.790265,-0.82005
Recovery Adjustments,1.573313,1.010585,0.772464,0.670097
LTV Adjustments,1.153641,1.484573,1.637395,1.041599
APR Adjustments,1.560127,0.932848,1.37773,1.448682
RAGU Score,146.917582,146.873167,148.8071,148.134688
LTV,1.522828,1.504595,1.496321,1.529102



Exported to: ../output/new_kmx_models.xlsx
[PROGRESS] Export Complete


In [10]:
print("="*80)
print("  VERIFICATION: All KMX vs bareboned_ragu_new.ipynb")
print("="*80)

all_kmx_df = results_by_model.get('All KMX')
if all_kmx_df is not None and len(all_kmx_df) > 0:
    try:
        reference_df = pd.read_csv('../output/all_df.csv')
        ref_kmx = reference_df[reference_df.lob == 'KMX'].copy() if 'lob' in reference_df.columns else pd.DataFrame()

        if len(ref_kmx) > 0:
            our_kmx = all_kmx_df.reset_index()
            our_kmx = our_kmx[our_kmx.lob == 'KMX'][['vintage', 'ms_original', 'gross_loss_impact', 'recovery_impact', 'ragu_score', 'loss_multiplier']].copy()
            our_kmx = our_kmx.set_index('vintage')

            ref_cols = ['vintage', 'ms_original', 'ragu_score', 'loss_multiplier']
            ref_available = [c for c in ref_cols if c in ref_kmx.columns]
            ref_kmx_compare = ref_kmx[ref_available].copy()
            ref_kmx_compare = ref_kmx_compare.set_index('vintage')

            common_vintages = sorted(set(our_kmx.index) & set(ref_kmx_compare.index))

            if common_vintages:
                comparison = pd.DataFrame(index=common_vintages)
                for col in [c for c in ['ragu_score', 'loss_multiplier'] if c in ref_kmx_compare.columns]:
                    comparison[f'{col}_ours'] = our_kmx.loc[common_vintages, col].values
                    comparison[f'{col}_ref'] = ref_kmx_compare.loc[common_vintages, col].values
                    comparison[f'{col}_diff'] = comparison[f'{col}_ours'] - comparison[f'{col}_ref']

                print(f"\nComparing {len(common_vintages)} common vintages:")
                display(comparison)
            else:
                print("No common vintages found.")
        else:
            print("No KMX rows found in reference all_df.csv.")
    except FileNotFoundError:
        print("all_df.csv not found. Run bareboned_ragu_new.ipynb first to generate the reference file.")
    except Exception as e:
        print(f"Error loading reference data: {e}")
        print("\nManually compare the 'All KMX' sheet in new_kmx_models.xlsx")
        print("against the KMX rows in barebones_ragu.xlsx")
else:
    print("No 'All KMX' results to verify.")

# Summary table
print("\n" + "="*80)
print("  SUMMARY: RAGU Scores by MTN Model (most recent vintage)")
print("="*80)
summary_rows = []
baseline_ltv = BASELINES['KMX']['ltv']
for mtn_model_filter in MTN_MODELS + ['All KMX']:
    label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
    model_df = results_by_model.get(mtn_model_filter)
    if model_df is not None and len(model_df) > 0:
        kmx_rows = model_df.reset_index()
        kmx_rows = kmx_rows[kmx_rows.lob == 'KMX']
        if len(kmx_rows) > 0:
            last_vintage = kmx_rows.vintage.max()
            last_row = kmx_rows[kmx_rows.vintage == last_vintage].iloc[0]
            summary_rows.append({
                'Model': label,
                'Latest Vintage': last_vintage,
                'Contract MS': last_row.get('ms_original', float('nan')),
                'Gross Loss Impact': last_row.get('gross_loss_impact', float('nan')),
                'Recovery Impact': last_row.get('recovery_impact', float('nan')),
                'LTV Impact': last_row.get('ltv_impact', float('nan')),
                'APR Impact': last_row.get('apr_impact', float('nan')),
                'RAGU Score': last_row.get('ragu_score', float('nan')),
                'LTV': last_row.get('ltv', float('nan')),
                'Loss Multiplier': last_row.get('loss_multiplier', float('nan')),
                'N Vintages': len(kmx_rows),
            })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows).set_index('Model')
    display(summary_df)

    print(f"\nRAGU decomposition:")
    print(f"  baseline_ltv = {baseline_ltv}")
    print(f"  baseline_apr = {BASELINES['KMX']['apr']}")
    print(f"  RAGU Score = (unit_loss * recovery * baselined_recovery) + ltv_impact + apr_impact")
else:
    print("No results available.")

  VERIFICATION: All KMX vs bareboned_ragu_new.ipynb
No common vintages found.

  SUMMARY: RAGU Scores by MTN Model (most recent vintage)


,Latest Vintage,Contract MS,Gross Loss Impact,Recovery Impact,LTV Impact,APR Impact,RAGU Score,LTV,Loss Multiplier,N Vintages
Model,,,,,,,,,,
MTN 3.0,2026 Q3,145.598432,5.474959,2.999967,3.764130,1.887093,159.724581,1.389954,0.781002,13
MTN 3.1,2026 Q2,142.989623,0.070744,0.565060,0.316308,0.985432,144.927166,1.571000,0.997170,7
MTN 3.2,2026 Q3,142.909828,0.397136,0.780448,1.424252,0.828788,146.340452,1.507886,0.984115,3
MTN 4.1,2026 Q3,145.794362,-0.820050,0.670097,1.041599,1.448682,148.134688,1.529102,1.032802,4
All KMX,2026 Q3,143.525531,0.146563,0.760866,1.345625,0.963382,146.741967,1.512197,0.994137,15



RAGU decomposition:
  baseline_ltv = 1.59
  baseline_apr = 0.25
  RAGU Score = (unit_loss * recovery * baselined_recovery) + ltv_impact + apr_impact
